# FUNSD — nhận dạng thực thể tài liệu — bản thực hành cơ bản

Primary Entity F1 ↑. Gold linking không có trong nielsr/funsd: K-V F1 không xác định. Bản basic dùng word/layout và vanilla RNN; toàn bộ scratch.

Chạy độc lập: **Restart Kernel → Run All**. Các trọng số ngôn ngữ/tokenizer đều học từ train. Không đọc notebook khác, fewshot hay workbook. Chưa huấn luyện đầy đủ; không kèm điểm benchmark đã đo.

**Bản basic cập nhật:** Bỏ đặc trưng word/char/ảnh không được model basic sử dụng; giữ offset/alignment, box input cần thiết và mask. Không phải bản tối ưu; không cam kết chênh lệch điểm.

**Cách viết training/checkpoint:** training loop được viết trực tiếp trong cell (sklearn dùng fit); lưu gọn khi validation cải thiện. Checkpoint phục vụ inference, không lưu optimizer/scheduler/RNG để resume.


## 1. Chuẩn bị thư viện

Dùng PyTorch/torchvision của kernel và datasets. Không dùng LayoutLM hoặc tokenizer pretrained.


In [ ]:
import importlib.util
import subprocess
import sys
NEEDED_PACKAGES = [('datasets', 'datasets==4.8.5')]
missing = [package for module, package in NEEDED_PACKAGES if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    raise RuntimeError("Cần kernel có PyTorch và torchvision tương thích; cài bản CPU/CUDA phù hợp rồi chạy lại.")

from pathlib import Path
from collections import Counter, defaultdict
import csv
import hashlib
import io
import json
import math
import random
import re
import unicodedata
import numpy as np
from PIL import Image, ImageOps
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm


## 2. Cấu hình cố định

Tất cả scratch. GPU 32 GB là đích thiết kế, không phải thiết bị bắt buộc. Không thay cấu hình theo VRAM.


In [ ]:
SEED = 42
VARIANT = "basic"
ARTIFACT_DIR = Path("artifacts/document_funsd") / VARIANT
CACHE_DIR = ARTIFACT_DIR / "data_cache"
WEIGHTS_CACHE_DIR = ARTIFACT_DIR / "weights"
CHECKPOINT_PATH = ARTIFACT_DIR / "best_model.pt"
PRETRAINING_POLICY = "scratch"
PRETRAINED_COMPONENTS = []
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(WEIGHTS_CACHE_DIR))  # Chỉ đặt nơi cache, không dùng torch.hub.load.
print(f"Thiết bị: {DEVICE}; biến thể: {VARIANT}; PyTorch {torch.__version__}; torchvision {torchvision.__version__}")


SOURCE = {'row': 4, 'task': 'FUNSD conditional entity recognition, không có gold K-V links', 'dataset': 'nielsr/funsd', 'revision': '7e7eeeedd84ce86540eb83cbbf7c75a3fcc7c7a5', 'url': 'https://huggingface.co/datasets/nielsr/funsd'}
DATASET_ID = SOURCE["dataset"]
DATASET_REVISION = SOURCE["revision"]
USE_LOCAL_DATA = False
LOCAL_DATA_DIR = Path("data/funsd_saved")
VAL_FRACTION = 0.20
IMAGE_SIZE = 256
MAX_WORD_CHARS = 32
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
HIDDEN_DIM = 64
DROPOUT = 0.2
EPOCHS = 60
PATIENCE = 12
LEARNING_RATE = 0.001
BACKBONE_LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
GRAD_CLIP = 1.0
PRIMARY_METRIC = "entity_f1"
DIRECTION = "max"


In [ ]:
def fingerprint(records):
    payload = json.dumps(records, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()

def open_rgb(value):
    if isinstance(value, Image.Image):
        return ImageOps.exif_transpose(value).convert("RGB")
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return ImageOps.exif_transpose(Image.open(io.BytesIO(value["bytes"]))).convert("RGB")
        value = value.get("path")
    if value is None:
        raise ValueError("Thiếu ảnh; cần PIL.Image, bytes ảnh hoặc đường dẫn thật.")
    with Image.open(value) as image:
        return ImageOps.exif_transpose(image).convert("RGB")

def to_device(batch):
    return {key: value.to(DEVICE) if torch.is_tensor(value) and key not in ("lengths", "widths", "target_lengths") else value
            for key, value in batch.items()}

def make_loader(dataset, train=False):
    return DataLoader(dataset, batch_size=BATCH_SIZE if train else EVAL_BATCH_SIZE,
                      shuffle=train, num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(SEED))


## 3. Đọc FUNSD và chia theo tài liệu

Cần OCR words/bboxes có sẵn; đây là nhận dạng thực thể có điều kiện, không phải OCR từ ảnh đơn lẻ. Tách 20% tài liệu train làm validation; test giữ nguyên. Không dùng ID HF đơn độc để chứng minh không trùng.


In [ ]:
from datasets import load_dataset, load_from_disk, Image as HFImage
try:
    raw = load_from_disk(str(LOCAL_DATA_DIR)) if USE_LOCAL_DATA else load_dataset(
        DATASET_ID, revision=DATASET_REVISION, cache_dir=str(CACHE_DIR))
except Exception as exc:
    raise RuntimeError(f"Không tải được nielsr/funsd. Kiểm tra mạng hoặc bật USE_LOCAL_DATA với "
                       f"DatasetDict save_to_disk đúng nguồn tại {LOCAL_DATA_DIR}. {exc}") from exc
if set(raw) != {"train", "test"}:
    raise ValueError("FUNSD cần đúng train/test; không lấy test để tạo validation.")
expected_tags = ["O", "B-HEADER", "I-HEADER", "B-QUESTION", "I-QUESTION", "B-ANSWER", "I-ANSWER"]
actual_tags = raw["train"].features["ner_tags"].feature.names
if actual_tags != expected_tags:
    raise ValueError(f"Label mapping khác nguồn đã kiểm tra: {actual_tags}")
TAG_NAMES = expected_tags

def canonical_bio(tags):
    result, previous = [], "O"
    for index in tags:
        name = TAG_NAMES[index]
        if name.startswith("I-") and previous not in ("B-" + name[2:], name):
            name = "B-" + name[2:]
        result.append(TAG_NAMES.index(name))
        previous = name
    return result

def adapt(dataset, split):
    required = {"id", "image", "words", "bboxes", "ner_tags"}
    if not required.issubset(dataset.column_names):
        raise ValueError(f"Thiếu trường FUNSD: {required - set(dataset.column_names)}")
    dataset = dataset.cast_column("image", HFImage(decode=False))
    rows, repairs = [], 0
    for row in dataset:
        if not row["words"] or not (len(row["words"]) == len(row["bboxes"]) == len(row["ner_tags"])):
            raise ValueError("words/bboxes/ner_tags không cùng độ dài hoặc rỗng.")
        for box in row["bboxes"]:
            if len(box) != 4 or not all(0 <= value <= 1000 for value in box) or box[0] > box[2] or box[1] > box[3]:
                raise ValueError(f"BBox không hợp lệ: {box}")
        tags = canonical_bio(row["ner_tags"])
        repairs += sum(a != b for a, b in zip(tags, row["ner_tags"]))
        # ID HF chỉ có ý nghĩa trong split. Không sắp lại words theo bbox vì làm đổi chuỗi nhãn.
        rows.append({"id": f"{split}:{row['id']}", "image": row["image"],
                     "words": row["words"], "boxes": row["bboxes"], "tags": tags})
    print(f"{split}: {len(rows)} tài liệu; chuẩn hóa {repairs} I-tag không hợp lệ thành B cùng loại.")
    return rows

all_train = sorted(adapt(raw["train"], "train"), key=lambda row: row["id"])
test_records = adapt(raw["test"], "test")
order = list(range(len(all_train)))
random.Random(SEED).shuffle(order)
valid_ids = set(order[:max(1, math.ceil(VAL_FRACTION * len(order)))])
train_records = [row for i, row in enumerate(all_train) if i not in valid_ids]
val_records = [row for i, row in enumerate(all_train) if i in valid_ids]
if not train_records or not val_records:
    raise ValueError("Không đủ tài liệu để chia train/validation.")
# So trùng bằng nội dung word+box, không chỉ ID HF được đánh lại theo split.
def document_key(row):
    return fingerprint({"words": row["words"], "boxes": row["boxes"]})
keys = [{document_key(r) for r in records} for records in (train_records, val_records, test_records)]
if keys[0] & keys[1] or keys[0] & keys[2] or keys[1] & keys[2]:
    raise ValueError("Có tài liệu trùng nội dung giữa các split; cần xử lý trước huấn luyện.")
SPLIT_FINGERPRINTS = {name: fingerprint([{k: r[k] for k in ("id", "words", "boxes", "tags")} for r in records])
                      for name, records in (("train", train_records), ("validation", val_records), ("test", test_records))}
print("Số tài liệu train/validation/test:", len(train_records), len(val_records), len(test_records))
print("Fingerprint:", SPLIT_FINGERPRINTS)
print("Không có nhãn linking trong schema này: K-V linking F1 sẽ là null, không phải 0.")


## 4. Vocabulary, layout và Dataset

Fit vocabulary chỉ train. Giữ nguyên word order và toàn bộ tài liệu, padding có mask. Chỉ rút ký tự trong đặc trưng word về 32, không bỏ word/nhãn. Box chuẩn hóa 0..1000.


In [ ]:
word_vocab = {"<PAD>": 0, "<UNK>": 1}
word_vocab.update({word: i + 2 for i, word in enumerate(sorted({w.lower() for r in train_records for w in r["words"]}))})

TOKENIZER_STATE = {"word_vocab": word_vocab, "tag_names": TAG_NAMES}
PREPROCESS_CONFIG = {"image_size": IMAGE_SIZE, "max_word_chars": MAX_WORD_CHARS,
                     "word_order": "giữ thứ tự HF", "boxes_scale": 1000,
                     "image_norm": "RGB 0..1", "bio_policy": "I sai ngữ cảnh thành B cùng loại"}
def restore_tokenizer(state):
    global word_vocab, TAG_NAMES
    word_vocab, TAG_NAMES = state["word_vocab"], state["tag_names"]

class FormDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        return self.records[index]

def collate_batch(items):
    lengths=torch.tensor([len(record["words"]) for record in items])
    width=int(lengths.max())
    words=torch.zeros(len(items),width,dtype=torch.long)
    boxes=torch.zeros(len(items),width,4)
    tags=torch.zeros(len(items),width,dtype=torch.long)
    for i,record in enumerate(items):
        n=len(record["words"])
        words[i,:n]=torch.tensor([word_vocab.get(w.lower(),1) for w in record["words"]])
        boxes[i,:n]=torch.tensor(record["boxes"],dtype=torch.float32)/1000
        tags[i,:n]=torch.tensor(record["tags"])
    return {"words":words,"boxes":boxes,"tags":tags,"lengths":lengths,
            "mask":torch.arange(width)[None]<lengths[:,None],
            "ids":[r["id"] for r in items],"records":items}

train_loader = make_loader(FormDataset(train_records), train=True)
val_loader = make_loader(FormDataset(val_records))
test_loader = make_loader(FormDataset(test_records))
print("Vocabulary train:", len(word_vocab), "từ;", 0, "đặc trưng ký tự; không đọc ảnh vì model basic không dùng.")


## 5. Kiến trúc

Word+layout, RNN một chiều, softmax có trọng số train. Ảnh và đặc trưng char không được dùng trong forward basic.


In [ ]:
class FormModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.word_embedding = nn.Embedding(cfg["n_words"], 64, padding_idx=0)
        self.layout = nn.Sequential(nn.Linear(6,16),nn.ReLU())
        self.encoder = nn.RNN(80,cfg["hidden_dim"],batch_first=True)
        self.dropout = nn.Dropout(cfg["dropout"])
        self.head = nn.Linear(cfg["hidden_dim"],len(TAG_NAMES))
        self.register_buffer("tag_weights", torch.ones(len(TAG_NAMES)))
    def forward(self, batch):
        box = batch["boxes"]
        layout = self.layout(torch.cat([box,box[:,:,2:]-box[:,:,:2]],-1))
        x = self.dropout(torch.cat([self.word_embedding(batch["words"]),layout],-1))
        packed = pack_padded_sequence(x,batch["lengths"],batch_first=True,enforce_sorted=False)
        hidden,_ = self.encoder(packed)
        hidden,_ = pad_packed_sequence(hidden,batch_first=True,total_length=batch["words"].size(1))
        return self.head(self.dropout(hidden))
    def loss(self, emissions, batch):
        return nn.functional.cross_entropy(emissions[batch["mask"]],batch["tags"][batch["mask"]],weight=self.tag_weights)
    def decode(self, emissions, batch):
        return [canonical_bio(ids[:length]) for ids,length in zip(emissions.argmax(-1).tolist(),batch["lengths"].tolist())]
def build_model(cfg, initialize_pretrained=True):
    model = FormModel(cfg)
    counts = torch.bincount(torch.tensor([t for r in train_records for t in r["tags"]]),minlength=len(TAG_NAMES)).float()
    weights = counts.clamp_min(1).rsqrt()
    model.tag_weights.copy_(weights / weights.mean())
    return model


In [ ]:
MODEL_CONFIG = {"n_words":len(word_vocab),"hidden_dim":HIDDEN_DIM,"dropout":DROPOUT}


## 6. Entity span F1

Exact entity micro-F1 với khóa doc/start_word/end_word/type, bỏ O/PAD. Cùng quy tắc I bất hợp lệ thành B cùng loại. Linking F1 trả null vì không có gold, không tự coi gần nhau là liên kết thật.


In [ ]:
def entity_spans(tags):
    tags = canonical_bio(tags)
    output, start, kind = [], None, None
    for i, tag_id in enumerate(tags + [0]):
        tag = TAG_NAMES[tag_id]
        if start is not None and (tag == "O" or tag.startswith("B-") or tag[2:] != kind):
            output.append((start,i,kind)); start,kind = None,None
        if tag.startswith("B-"):
            start,kind = i,tag[2:]
    return output

def compute_loss(model,batch):
    return model.loss(model(batch),batch)

@torch.no_grad()
def evaluate(model,loader,return_predictions=False):
    model.eval()
    gold,predicted=set(),set()
    examples=[]
    for cpu_batch in loader:
        batch=to_device(cpu_batch)
        paths=model.decode(model(batch),batch)
        for sid,record,path in zip(batch["ids"],batch["records"],paths):
            gold.update((sid,a,b,k) for a,b,k in entity_spans(record["tags"]))
            guessed=entity_spans(path)
            predicted.update((sid,a,b,k) for a,b,k in guessed)
            if return_predictions:
                examples.append({"id":sid,"entities":[{"start_word":a,"end_word":b,"type":k,
                    "text":" ".join(record["words"][a:b]),"word_boxes":record["boxes"][a:b]} for a,b,k in guessed],
                    "key_value_links":None,"linking_status":"Nguồn HF không có gold linking"})
    tp=len(gold & predicted)
    precision=tp/len(predicted) if predicted else 0.0
    recall=tp/len(gold) if gold else 0.0
    score=2*tp/(len(gold)+len(predicted)) if gold or predicted else 0.0
    metrics={"entity_f1":score,"precision":precision,"recall":recall,"tp":tp,
             "gold_entities":len(gold),"predicted_entities":len(predicted),"key_value_linking_f1":None}
    return (metrics,examples) if return_predictions else metrics


## 7. Huấn luyện và checkpoint

AdamW, cosine, clipping; early stopping theo Entity F1 validation. Hai mô hình có ngân sách đầy đủ riêng, không có bài tập thiếu mã.


In [ ]:


def make_optimizer(model):
    visual, remaining = [], []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            (visual if name.startswith("backbone.") else remaining).append(parameter)
    groups = [{"params": remaining, "lr": LEARNING_RATE}]
    if visual:
        groups.append({"params": visual, "lr": BACKBONE_LEARNING_RATE})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)




In [ ]:
model = build_model(MODEL_CONFIG, initialize_pretrained=True).to(DEVICE)
print(f"Tham số học: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
optimizer = make_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_score = math.inf if DIRECTION == "min" else -math.inf
stale_epochs = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total, seen = 0.0, 0
    for cpu_batch in tqdm(train_loader, desc="Huấn luyện", leave=False):
        batch = to_device(cpu_batch)
        optimizer.zero_grad(set_to_none=True)
        loss = compute_loss(model, batch)
        if not torch.isfinite(loss):
            raise FloatingPointError("Loss không hữu hạn; kiểm tra nhãn, mask và cấu hình cố định.")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        count = len(batch["ids"])
        total += loss.item() * count
        seen += count
    if not seen:
        raise ValueError("Tập huấn luyện rỗng.")
    loss = total / seen
    metrics = evaluate(model, val_loader)
    score = float(metrics[PRIMARY_METRIC])
    if not math.isfinite(score):
        raise ValueError("Metric validation không hữu hạn.")
    scheduler.step()
    print(f"Epoch {epoch:02d}: loss={loss:.4f}; validation={metrics}")
    improved = score < best_score if DIRECTION == "min" else score > best_score
    if improved:
        best_score, stale_epochs = score, 0
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "tokenizer": TOKENIZER_STATE,
        }, CHECKPOINT_PATH)
        print("Đã lưu checkpoint tốt nhất theo primary metric validation.")
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Dừng sớm sau {PATIENCE} epoch không cải thiện.")
            break


## 8. Reload và đánh giá test

Tải toàn pipeline từ checkpoint mới rồi chấm test, xuất entities và trạng thái thiếu linking. Không tạo điểm K-V linking giả.


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
assert checkpoint["tokenizer"] == TOKENIZER_STATE
restore_tokenizer(checkpoint["tokenizer"])
# Không tải backbone lần nữa: state_dict chứa đầy đủ trọng số visual và head.
# Cấu hình kiến trúc ở notebook; kích thước vocab được lấy sau khi restore.
inference_config = {"n_words":len(word_vocab),"hidden_dim":HIDDEN_DIM,"dropout":DROPOUT}
best_model = build_model(inference_config, initialize_pretrained=False).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state"])
best_model.eval()
validation_metrics = evaluate(best_model, val_loader)
test_metrics, predictions = evaluate(best_model, test_loader, return_predictions=True)
results = {"epoch": checkpoint["epoch"], "source": SOURCE, "split_fingerprints": SPLIT_FINGERPRINTS,
           "validation": validation_metrics, "held_out_test": test_metrics}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "predictions.json").write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Đã nạp epoch {checkpoint['epoch']}.")
print("Validation:", validation_metrics)
print("Held-out:", test_metrics)
print("Ví dụ dự đoán:", predictions[0] if predictions else "Không có mẫu.")


## 9. Phân tích tài liệu mới

Hàm cần image, words, bboxes; trả thực thể theo word offsets. Muốn dùng ảnh thô cần thêm OCR riêng chưa có trong pipeline hiện tại.


In [ ]:
@torch.no_grad()
def parse_form(image,words,bboxes):
    if not words or len(words)!=len(bboxes):
        raise ValueError("Cần words và bboxes chuẩn hóa 0..1000, cùng độ dài; ảnh đơn lẻ chưa đủ cho pipeline này.")
    for box in bboxes:
        if len(box)!=4 or not all(0<=v<=1000 for v in box) or box[0]>box[2] or box[1]>box[3]:
            raise ValueError("BBox sai định dạng/miền giá trị.")
    record={"id":"new","image":image,"words":words,"boxes":bboxes,"tags":[0]*len(words)}
    dataset=FormDataset([record])
    batch=to_device(collate_batch([dataset[0]]))
    best_model.eval()
    tags=best_model.decode(best_model(batch),batch)[0]
    return [{"start_word":a,"end_word":b,"type":kind,"text":" ".join(words[a:b])}
            for a,b,kind in entity_spans(tags)]

sample=test_records[0]
print(parse_form(sample["image"],sample["words"],sample["boxes"]))
